# Machine learning: teaching a computer to recognise a genre

## What is different here

Everything we have done so far, we told the computer how to do. We wrote the rule: *count this word*, *divide by that length*, *sort*.

Machine learning turns that around. We give the computer **examples with the right answer attached**, and it works out the rule itself.

Our examples are the 1,750 British Library books from the last notebook. Each one has a title, and each one was labelled *Fiction* or *Non-fiction* by a human annotator. The question: **can a computer predict the label from the title alone?**

Two pieces of vocabulary, and they are the only new ones:
* **features** — what the model gets to look at (here: the words in the title)
* **label** — what it has to predict (here: the genre)

Supervised learning is nothing more than: *learn the relationship between features and labels from examples, then apply it to examples you have never seen*.

In [ ]:
!wget -q https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/main/Sessions/data/bl_books_genre.csv

In [ ]:
import pandas as pd

df = pd.read_csv("bl_books_genre.csv")
print(df.shape)
df.head(3)

## Preparing the data

Two decisions before we start, both of which matter more than any model choice.

**First**, we keep only the books labelled *Fiction* or *Non-fiction*, dropping the handful marked "Both" or "Can't tell".

**Second**, we keep only the **English** books. Remember what we found in the last notebook: the Danish and Swedish books are almost all non-fiction. If we left them in, a model could score well by learning "Scandinavian word ⇒ non-fiction" — which is not genre recognition at all. We come back to this at the end.

In [ ]:
books = df[df["genre"].isin(["Fiction", "Non-fiction"])]
books = books[books["language"] == "English"]

print("books:", len(books))
print(books["genre"].value_counts())

## Splitting into training and test data

This is the rule that makes machine learning trustworthy, and it is easy to state:

> **Never evaluate a model on the examples it learned from.**

A model that has seen an example can simply memorise its answer. To find out whether it has learned anything general, we hide some data from it and test on that.

`train_test_split` does this for us. `stratify=y` keeps the fiction/non-fiction proportion the same in both halves, and `random_state=42` makes the split reproducible — you and your neighbour will get identical results.

In [ ]:
from sklearn.model_selection import train_test_split

X = books["title"]                            # the features: the raw titles
y = (books["genre"] == "Fiction").astype(int) # the label: 1 for fiction, 0 for non-fiction

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("training examples:", len(X_train))
print("test examples:    ", len(X_test))
print("fiction in test set:", f"{y_test.mean():.0%}")

## From titles to numbers

A model cannot read. It needs numbers — and we already know how to turn a text into numbers, because that is exactly what we did in notebook 2g: **count the words**.

`TfidfVectorizer` does the whole of notebook 2g in one line. It builds the vocabulary, counts the words in every document, divides by document length (term frequency) and weights each word by how rare it is across the collection (inverse document frequency).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

print("shape of the training matrix:", X_train_vectors.shape)

Read that shape carefully: **630 rows** — one per book — and **2,210 columns, one per word in the vocabulary**.

Every book is now a row of numbers, almost all of them zero, with values only in the columns for the words its title happens to use. This is the **document-term matrix**: the bag-of-words idea from day 2, written out as a table.

⚠️ Note the two different method names, because this trips everybody up:
* `.fit_transform()` on the **training** data: work out the vocabulary *and* convert.
* `.transform()` on the **test** data: convert using the vocabulary already learnt.

If we called `.fit_transform()` on the test set, it would build its vocabulary from data the model is not allowed to see. The test set must stay sealed.

## Model 1: k-nearest neighbours

You have already written this model. In notebook 2g you took a query, found the documents most similar to it by cosine similarity, and returned the top few.

**k-nearest neighbours** does exactly that, and then adds one step: the neighbours *vote*.

To classify a book we have never seen: find the `k` most similar titles among the ones we do have labels for, and give it whatever label the majority of them have. There is no training in any real sense — the model just keeps the examples.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
knn.fit(X_train_vectors, y_train)

predictions = knn.predict(X_test_vectors)
print(predictions[:20])

`1` means fiction, `0` non-fiction. How many did it get right?

In [ ]:
from sklearn.metrics import accuracy_score

print("accuracy:", round(accuracy_score(y_test, predictions), 3))

### Looking at the neighbours

Because k-NN is just "find the most similar examples", we can ask it **which** books it consulted. Very few models let you do this, and it is worth taking advantage of.

In [ ]:
labels = ["Non-fiction", "Fiction"]
training_titles = list(X_train)

def show_neighbours(title, k=5):
    """ print the k nearest training titles for a given title, and the prediction """
    vector = vectorizer.transform([title])
    distances, indices = knn.kneighbors(vector, n_neighbors=k)

    print(f"QUERY: {title}")
    print(f"   -> predicted: {labels[knn.predict(vector)[0]]}")
    for distance, index in zip(distances[0], indices[0]):
        similarity = 1 - distance
        print(f"   similarity={similarity:.2f}  [{labels[y_train.iloc[index]]:11s}] {training_titles[index][:60]}")


show_neighbours("The Poems of Kent")

### ✏️ Exercise 1

Try `show_neighbours()` on a few titles of your own invention — some that sound like novels, some like local histories. Can you find a title it gets obviously wrong? Look at the neighbours it used: can you see *why* it went wrong?

In [ ]:
# Type your code here:


### Choosing k

`k` is a **hyperparameter**: a setting we choose, rather than something the model learns. Let's try a few values.

In [ ]:
for k in [1, 3, 5, 15, 50]:
    model = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    model.fit(X_train_vectors, y_train)
    score = accuracy_score(y_test, model.predict(X_test_vectors))
    print(f"k={k:3d}  accuracy={score:.3f}")

There is a sweet spot in the middle, and both extremes are bad for reasons you can reason about without any mathematics:

* **k=1** trusts a single nearest book. One eccentric title in the training data and the answer flips.
* **k=50** averages over so many books that the answer drifts towards "whatever is most common overall".

Small k pays too much attention to individual examples; large k pays too little. Nearly every knob in machine learning is some version of this trade-off.

## Is 0.9 good? The baseline

An accuracy on its own means nothing. You always need something to compare it against.

The simplest comparison: a "model" that ignores the title completely and always answers with whichever label is most common.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_vectors, y_train)

print("baseline accuracy:", round(accuracy_score(y_test, baseline.predict(X_test_vectors)), 3))

Our corpus is 62% non-fiction, so a model that always says "non-fiction" is right 62% of the time while knowing nothing whatsoever.

**This is why accuracy alone is a dangerous number.** Had the imbalance been sharper — 95% non-fiction, as it would be in many real collections — that useless model would report 95% accuracy.

## Measuring properly: precision, recall, F1

We need to know how the model does on **each class**, not just overall. Two questions, and they are different:

* **Precision**: when the model says "fiction", how often is it right? *(Do we trust its claims?)*
* **Recall**: of all the fiction that is really there, how much did it find? *(Does it miss things?)*

**F1** combines the two into a single number. The **confusion matrix** shows the raw counts behind them.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions, target_names=["Non-fiction", "Fiction"]))

Read the confusion matrix as:

```
              predicted non-fiction   predicted fiction
actually non-fiction      correct            wrong
actually fiction           wrong            correct
```

A researcher building a corpus of Victorian novels cares far more about **recall** — the novels the model failed to find are invisible in everything they do afterwards.

## Model 2: a support vector machine

k-NN never really learns anything: it stores the examples and compares. A **support vector machine** does something different — it looks for the **boundary** that best separates fiction from non-fiction, keeping as much clear space as possible on either side.

Then a new book is classified by which side of that boundary it falls on.

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_vectors, y_train)

svm_predictions = svm.predict(X_test_vectors)

print("k-NN accuracy:", round(accuracy_score(y_test, predictions), 3))
print("SVM accuracy: ", round(accuracy_score(y_test, svm_predictions), 3))
print()
print(classification_report(y_test, svm_predictions, target_names=["Non-fiction", "Fiction"]))

### ✏️ Exercise 2

Above, we vectorised with **TF-IDF**. Go back and build `CountVectorizer()` instead — plain word counts, with no length normalisation and no weighting by rarity — and compare both models.

Which model suffers more without TF-IDF, and can you explain why? (Think about what k-NN actually measures.)

In [ ]:
# Type your code here:


## What did the model actually learn?

This is the part that should interest a humanist most. A linear SVM assigns a **weight** to every word: strongly positive words push a title towards fiction, strongly negative towards non-fiction.

We can simply read them off.

In [ ]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
weights = svm.coef_[0]

most_fiction = np.argsort(weights)[-15:][::-1]
most_nonfiction = np.argsort(weights)[:15]

print("pushes towards FICTION    :", list(feature_names[most_fiction]))
print()
print("pushes towards NON-FICTION:", list(feature_names[most_nonfiction]))

No one told the model that *novel*, *tale*, *poems* and *romance* signal fiction, or that *history*, *sketches* and *illustrated* signal non-fiction. It found that in 630 titles.

That list is a research finding in miniature: it is a description, drawn from evidence, of how nineteenth-century books announced their genre on the title page.

### ✏️ Exercise 3: what did it *really* learn?

Now go back and remove the line that filtered to English books, so that all 1,744 labelled books are used. Retrain, and look at the strongest non-fiction weights again.

Among them you will find `og`, `af`, `och`, `van`, `del` — Danish, Swedish and Dutch function words.

What has the model actually learned? And is its accuracy on that version a fair measure of how well it recognises *genre*? This is not a bug in the code. It is a property of the collection, of a kind that is very easy to publish by accident.

In [ ]:
# Type your code here:


## Where this breaks — and where we go next

Our model is good. Now let's break it on purpose, because how a method fails tells you what it is really doing.

Below are pairs of titles. Each pair means nearly the same thing; only one word differs.

In [ ]:
def fiction_score(title):
    """ how confidently does the SVM call this fiction? above 0 = fiction """
    return svm.decision_function(vectorizer.transform([title]))[0]


pairs = [
    ("A Tale of Cornwall",  "A Saga of Cornwall"),
    ("The Poems of Kent",   "The Sonnets of Kent"),
    ("A Novel of Kent",     "A Novella of Kent"),
]

for original, swapped in pairs:
    a, b = fiction_score(original), fiction_score(swapped)
    print(f"{original:22s} {a:+.2f} {labels[int(a > 0)]:12s}   {swapped:22s} {b:+.2f} {labels[int(b > 0)]}")

Swapping *tale* for *saga* — words a reader would treat as near-identical — moves a title from confidently fiction to the wrong side of the boundary.

Why? Look at the weights the model has for those words:

In [ ]:
for word in ["tale", "saga", "poems", "sonnets", "novel", "novella"]:
    if word in vectorizer.vocabulary_:
        print(f"{word:9s} weight = {weights[vectorizer.vocabulary_[word]]:+.2f}")
    else:
        print(f"{word:9s} NOT IN THE VOCABULARY — no column, so it counts for nothing")

There it is. `saga` and `novella` never appeared in the 630 training titles, so the model has **no column for them**. It did not judge them and decide they were weak evidence; it could not see them at all.

And this is not a rare edge case:

In [ ]:
import re

def words_in(texts):
    return set(word for text in texts for word in re.findall(r"[a-z]+", text.lower()))

train_vocabulary = words_in(X_train)
test_vocabulary = words_in(X_test)
unseen = test_vocabulary - train_vocabulary

print(f"{len(unseen)} of the {len(test_vocabulary)} words in the test titles never appeared in training")
print(f"that is {len(unseen) / len(test_vocabulary):.0%} of the test vocabulary, invisible to the model")

**Here is the limitation, stated plainly.** In a bag of words, every word is its own separate column, with no relationship to any other. The model has no way of knowing that a *novella* is a kind of *novel*, that *sonnets* are *poems*, or that a *saga* is a *tale*. Each is just column number 1,472 or column number 89.

To do better, we would need a representation in which **similar words are similar numbers**.

That representation exists, and you already have one in this repository — vectors trained on 1860s newspapers:

(To keep this quick, we are loading a small **extract** of those vectors — a few hundred words, enough for the comparisons below. Tomorrow we load the full model, all 48,054 words of it.)

In [ ]:
!wget -q https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/main/Sessions/data/1860s-vectors-sample.txt

from gensim.models import KeyedVectors

vectors = KeyedVectors.load_word2vec_format("1860s-vectors-sample.txt", binary=False)

print("similar meanings:")
for a, b in [("poems", "sonnets"), ("tale", "romance"), ("novel", "romance")]:
    print(f"   similarity({a}, {b}) = {vectors.similarity(a, b):.2f}")

print("\nunrelated words, for comparison:")
for a, b in [("poems", "railway"), ("novel", "boiler"), ("tale", "steam")]:
    print(f"   similarity({a}, {b}) = {vectors.similarity(a, b):.2f}")

print("\nnearest neighbours of 'sonnets':", [w for w, _ in vectors.most_similar("sonnets", topn=6)])

The exact words that broke our classifier — `poems` and `sonnets`, `tale` and `romance` — are **neighbours** in this space, while `poems` and `railway` are far apart. Nobody wrote those relationships down. They were learned from how the words are used.

**Where do such vectors come from, and what else can they do?** That is tomorrow.

### One honest warning about that last cell

Try `vectors.most_similar("novella")` and you will get nonsense — Italian names and unrelated words. The 1860s newspapers barely ever used the word, so its vector was learned from almost no evidence.

Word embeddings do not know language. They know the corpus they were trained on. A rare word gets a bad vector, and a corpus with a particular view of the world produces vectors that share it — something to keep firmly in mind tomorrow.

# Solutions

### ✏️ Exercise 1

In [ ]:
show_neighbours("A History of the Parish of Wingham")
print()
show_neighbours("The Mill on the Floss")
print()
# The classic failure: a fiction title whose words are all generic, so the neighbours
# are chosen by common words like "the" and "of" rather than by anything about genre.
show_neighbours("The Sonnets of Kent")

### ✏️ Exercise 2

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()
train_counts = count_vectorizer.fit_transform(X_train)
test_counts = count_vectorizer.transform(X_test)

knn_counts = KNeighborsClassifier(n_neighbors=5, metric="cosine").fit(train_counts, y_train)
svm_counts = LinearSVC().fit(train_counts, y_train)

print("with plain counts:")
print("   k-NN:", round(accuracy_score(y_test, knn_counts.predict(test_counts)), 3))
print("   SVM :", round(accuracy_score(y_test, svm_counts.predict(test_counts)), 3))
print("with TF-IDF:")
print("   k-NN:", round(accuracy_score(y_test, predictions), 3))
print("   SVM :", round(accuracy_score(y_test, svm_predictions), 3))

# k-NN suffers most. It works by measuring similarity between whole titles, so without
# IDF a match on "the" or "of" counts as much as a match on "novel" — and long titles
# look different from short ones for no good reason. The SVM can compensate by learning
# a small weight for common words, so it cares less.

### ✏️ Exercise 3

In [ ]:
all_books = df[df["genre"].isin(["Fiction", "Non-fiction"])]      # no language filter

Xa = all_books["title"]
ya = (all_books["genre"] == "Fiction").astype(int)
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    Xa, ya, test_size=0.25, random_state=42, stratify=ya
)

vec_all = TfidfVectorizer()
svm_all = LinearSVC().fit(vec_all.fit_transform(Xa_train), ya_train)

print("accuracy on all languages:", round(accuracy_score(ya_test, svm_all.predict(vec_all.transform(Xa_test))), 3))

names_all = np.array(vec_all.get_feature_names_out())
w_all = svm_all.coef_[0]
print("\npushes towards NON-FICTION:", list(names_all[np.argsort(w_all)[:15]]))

# `og`, `af`, `och`, `historisk` are Danish and Swedish. In this collection the
# Scandinavian books are almost all non-fiction, so "is this book in Danish?" is a
# very good predictor of non-fiction — and the model happily learned that instead
# of learning about genre. The accuracy looks fine. The model is not doing what we
# think it is doing. Always read the features.